# Step 0: Data Ingestion + EDA

**Goal:** Load data, understand structure, visualize trajectories

**Tasks:**
- [x] Load all 100 batches into unified structure
- [ ] Document batch lengths, sampling rates
- [ ] Identify missing values, anomalies
- [ ] Plot representative trajectories per control mode (DO2, Fs, P)
- [ ] Compute final P distribution per control mode

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.data_loader import load_batches, get_batch_info, get_final_penicillin, load_statistics

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Load all batches
batches = load_batches()
print(f"Loaded {len(batches)} batches")

# Show sample batch
batches[1].head()

In [ ]:
# Get batch info summary
batch_info = get_batch_info(batches)
batch_info.head(10)

## 2. Batch Lengths and Sampling Rates

In [ ]:
# Summary statistics by control mode
batch_info.groupby('control_mode')[['length', 'duration_h']].describe().round(2)

In [ ]:
# Visualize batch lengths by control mode
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=batch_info, x='control_mode', y='length', ax=axes[0])
axes[0].set_title('Batch Length (samples) by Control Mode')
axes[0].set_xlabel('Control Mode')
axes[0].set_ylabel('Number of Samples')

sns.boxplot(data=batch_info, x='control_mode', y='duration_h', ax=axes[1])
axes[1].set_title('Batch Duration (hours) by Control Mode')
axes[1].set_xlabel('Control Mode')
axes[1].set_ylabel('Duration (h)')

plt.tight_layout()
plt.savefig('../outputs/figures/batch_lengths.png', dpi=150)
plt.show()

In [ ]:
# Check sampling rate (should be 0.2h = 12 min)
sample_batch = batches[1]
time_diffs = sample_batch['time'].diff().dropna()
print(f"Sampling interval: {time_diffs.mean():.2f}h (std: {time_diffs.std():.4f})")
print(f"Expected: 0.2h (12 minutes)")

## 3. Missing Values and Anomalies

In [ ]:
# Check missing values across all batches
all_data = pd.concat(batches.values(), ignore_index=True)
missing_pct = (all_data.isnull().sum() / len(all_data) * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0]

In [ ]:
# Key input features for modeling
input_features = ['DO2', 'Fs', 'Fa', 'Fb', 'Fg', 'T', 'pH', 'RPM', 'CO2outgas', 'OUR', 'Fpaa']
target_features = ['P', 'P_offline']

# Check missing values in input features
available_inputs = [f for f in input_features if f in all_data.columns]
print("Input features available:", available_inputs)
print("\nMissing values in inputs:")
print(all_data[available_inputs].isnull().sum())

In [ ]:
# Identify fault batches
fault_batches = batch_info[batch_info['is_fault']]
print(f"Fault batches: {fault_batches['batch_id'].tolist()}")
print(f"Total fault batches: {len(fault_batches)}")

## 4. Representative Trajectories by Control Mode

In [ ]:
def plot_trajectories(batches, batch_ids, variable, title, ax):
    """Plot trajectories for selected batches."""
    for bid in batch_ids:
        df = batches[bid]
        ax.plot(df['time'], df[variable], alpha=0.7, label=f'Batch {bid}')
    ax.set_xlabel('Time (h)')
    ax.set_ylabel(variable)
    ax.set_title(title)
    ax.legend(loc='best', fontsize=8)

In [ ]:
# Select representative batches from each control mode (excluding faults)
recipe_batches = [1, 10, 20]  # 1-30
operator_batches = [35, 45, 55]  # 31-60
apc_batches = [65, 75, 85]  # 61-90

variables_to_plot = ['DO2', 'Fs', 'P']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for i, var in enumerate(variables_to_plot):
    plot_trajectories(batches, recipe_batches, var, f'{var} - Recipe (1-30)', axes[i, 0])
    plot_trajectories(batches, operator_batches, var, f'{var} - Operator (31-60)', axes[i, 1])
    plot_trajectories(batches, apc_batches, var, f'{var} - APC (61-90)', axes[i, 2])

plt.tight_layout()
plt.savefig('../outputs/figures/trajectories_by_control.png', dpi=150)
plt.show()

In [ ]:
# Plot all batches for key variables (colored by control mode)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = {'recipe': 'blue', 'operator': 'green', 'apc': 'orange', 'fault': 'red'}

for var, ax in zip(['DO2', 'Fs', 'P'], axes):
    for bid, df in batches.items():
        mode = batch_info[batch_info['batch_id'] == bid]['control_mode'].values[0]
        ax.plot(df['time'], df[var], color=colors[mode], alpha=0.3, linewidth=0.5)
    ax.set_xlabel('Time (h)')
    ax.set_ylabel(var)
    ax.set_title(f'{var} - All Batches')

# Add legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=c, label=m) for m, c in colors.items()]
axes[2].legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.savefig('../outputs/figures/all_trajectories.png', dpi=150)
plt.show()

## 5. Final Penicillin Distribution by Control Mode

In [ ]:
# Get final penicillin concentration
final_p = get_final_penicillin(batches)
final_p = final_p.merge(batch_info[['batch_id', 'control_mode']], on='batch_id')
final_p.head()

In [ ]:
# Statistics by control mode
final_p.groupby('control_mode')['final_P'].describe().round(2)

In [ ]:
# Visualize final P distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
sns.boxplot(data=final_p, x='control_mode', y='final_P', ax=axes[0])
axes[0].set_title('Final Penicillin Concentration by Control Mode')
axes[0].set_xlabel('Control Mode')
axes[0].set_ylabel('Final P (g/L)')

# Histogram
for mode in ['recipe', 'operator', 'apc', 'fault']:
    data = final_p[final_p['control_mode'] == mode]['final_P']
    axes[1].hist(data, bins=10, alpha=0.5, label=mode)
axes[1].set_title('Final P Distribution')
axes[1].set_xlabel('Final P (g/L)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/final_P_distribution.png', dpi=150)
plt.show()

## 6. Summary Statistics

In [ ]:
# Load and display batch statistics
stats = load_statistics()
stats.head(10)

In [ ]:
# Summary
print("=" * 50)
print("EDA SUMMARY")
print("=" * 50)
print(f"Total batches: {len(batches)}")
print(f"Fault batches (excluded): {len(fault_batches)}")
print(f"Valid batches: {len(batches) - len(fault_batches)}")
print(f"\nSampling rate: 0.2h (12 min)")
print(f"\nBatch lengths:")
print(batch_info.groupby('control_mode')['length'].agg(['min', 'max', 'mean']).round(0))
print(f"\nFinal P by control mode (g/L):")
print(final_p.groupby('control_mode')['final_P'].agg(['mean', 'std']).round(2))

## Key Observations

1. **Batch structure**: 100 batches total, 10 fault batches (91-100) to exclude
2. **Sampling**: Consistent 0.2h (12 min) intervals
3. **Variable lengths**: APC batches tend to be longer than recipe/operator
4. **Domain differences**: Visible trajectory differences between control modes
5. **Target distribution**: Final P varies by control mode - potential domain shift